# Kaggle Ablation Runner

This notebook stages the drought data in `/kaggle/working/`, writes the current project scripts from embedded source bundles, runs region clustering, launches ablation training jobs, and saves logs plus a metrics summary.

Attach the dataset that provides:

- `/kaggle/input/datasets/cheukhongtang/drought/train.csv`
- `/kaggle/input/datasets/cheukhongtang/drought/test.csv`
- `/kaggle/input/datasets/cheukhongtang/drought/sample_submission.csv`


In [1]:
# User settings
from pathlib import Path

INPUT_DIR = Path('/kaggle/input/datasets/cheukhongtang/drought')
WORK_DIR = Path('/kaggle/working')
DATA_DIR = WORK_DIR / 'data'
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
SUBMISSION_DIR = WORK_DIR / 'submissions'
LOG_DIR = WORK_DIR / 'logs'

# Choose the amount of data used for ablation training.
# Regions are sampled as whole units so each region's time windows stay valid.
TRAIN_FRACTION = 1
MAX_REGIONS = None
RECENT_YEARS_PER_REGION = 10

# Training budget for short ablation runs.
N_EPOCHS = 8
SAMPLES_PER_REGION_PER_EPOCH = 50
BATCH_SIZE = 256
MAX_PARALLEL_RUNS = 2
SEED = 42
SUBSET_TEST_TO_SELECTED_REGIONS = True

# Kaggle T4 x2: this script runs one independent training process per visible GPU.
GPU_IDS = ['0', '1']
NUM_WORKERS = 2

# Keep this false for ablation screening. Set true only for full-data final submissions.
USE_TTA = False

# For small subsets, reduce cluster min-size automatically so clustering does not fail.
AUTO_ADJUST_CLUSTERING_FOR_SUBSETS = True
CLUSTER_TARGET_K = None
CLUSTER_MIN_SIZE = None

# Turn individual ablations on/off here.
ABLATIONS = [
    {'name': 'baseline', 'purpose': 'Full current model', 'enabled': True, 'cfg': {}},
    {'name': 'no_daily', 'purpose': 'Remove 91-day Transformer branch', 'enabled': True,
     'cfg': {'use_daily_branch': 'false'}},
    {'name': 'no_long_context', 'purpose': 'Remove engineered long-context features', 'enabled': True,
     'cfg': {'use_long_context': 'false'}},
    {'name': 'no_cluster', 'purpose': 'Remove cluster features and embedding', 'enabled': True,
     'cfg': {'use_cluster_embedding': 'false', 'include_cluster_aggs': 'false', 'mixup_within_cluster_only': 'false'}},
    {'name': 'no_region_embedding', 'purpose': 'Remove region identity embedding', 'enabled': True,
     'cfg': {'use_region_embedding': 'false'}},
    {'name': 'replicate_weeks', 'purpose': 'Predict the same value for all 5 future weeks', 'enabled': True,
     'cfg': {'replicate_prediction': 'true'}},
    {'name': 'no_training_tricks', 'purpose': 'Remove mixup, label noise, and curriculum', 'enabled': True,
     'cfg': {'use_mixup': 'false', 'use_label_noise': 'false', 'use_curriculum': 'false'}},
    {'name': 'extra_features', 'purpose': 'Turn on extra engineered features', 'enabled': True,
     'cfg': {'extra_features': 'true'}},
]


In [2]:
# Imports and output folders
import base64
import gzip
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime

import numpy as np
import pandas as pd

for p in (DATA_DIR, CHECKPOINT_DIR, SUBMISSION_DIR, LOG_DIR):
    p.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}:', torch.cuda.get_device_name(i))
except Exception as exc:
    print('Torch/CUDA check failed:', exc)
print('Working dir:', WORK_DIR)


Python: 3.12.13
Torch: 2.10.0+cu128
CUDA available: True
CUDA device count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Working dir: /kaggle/working


## Write Project Scripts

The next cell reconstructs `deep_drought.py`, `cluster_regions.py`, and `postprocess_submission.py` inside `/kaggle/working/` from compressed embedded source. This makes the notebook self-contained.

In [3]:
# Embedded project source bundles. Generated from the local repository.
SCRIPT_BUNDLES = {
  "deep_drought.py": {
    "payload": "...",
    "sha256": "c6bf9cdea9b6567056cd337d0456a4d885805d4c95f0b44f304cdafdccb5770e",
    "bytes": 102862
  },
  "cluster_regions.py": {
    "payload": "H4sIAMO+ImoC/71a/Y7bNhL/f4F9ByIBKuoqa22naRtfHSCXbHpB0iRI0msujiFwJdomVqZ0krxed2vg/uoDFPeEfZKb4YdEyXKyvTuc0WZtfgyHM8PffJB37tyJ001Z8SIq+FJksgzzHfn9n/8icbbONxUncSrWrOJnSZFtlquKLDirNgUvSc4LoucQJpPTkyUMyE1LSbaiWpEtFzCFJ+QnViRkJXjBinglYpYSs6qQy/D05PTk1aaC1crJ6QmBjyYS2aXCuLwi6pNJTops66wdkIJtLVPkiqUbXraImIUcIqZDJGTw0PKBv9Ysz4EfQkdh+DxaCMlSv0Uq4RKksCzYOsxhHHzUvprmAKSWZsUg3lSEVeTdozffn7+LnuMGX4ucp0Jys8NRSN7wsipEXJGqYEIq9qqMpKysyItXr57/5dHj59Hfzx+9eevsNtSzxyF5bNQz+qpPI2YV9RlYFZILVioeCGVSblgarTmTUV7wWORnFV/nAcF/o4LJJQ8IK0Qiqp3folVyVmYgGegg+QoIEloKeRZnJckWZJ3JahVli2jNriOkRb4g+FWvcZwSSILLJVgMfZz9rSaUwhJqIlBBYu351iAv+IpdiWxTEGqaYAMV8K92V8ZZwaPtioP6dG+bil7g919/s+RKLktRiStki8LkgqesUoYms2IN7P4M9lytQN6rLE0MsXsh+TBQS8Gym7QSOfB+sSM/nT/7/q/v3gbKUAYg+0u25K5pGIV+FZIfeIFdcmdNkpSwWgp/qxWT5IdnL6PHL358++78TfT22YdzIiRYi6hKIjkDZqp6VxIP3YWSR1ateLEVoKQqyzQ9tDFlcISRTckXmxTNZmDXXGcJT33D1X00UrlZX0CHPUakORx47EnJrjgMv3PnzunJosjWJGegOHFBxDrPioq8hp+nJ+YH0AJ0YcBzXrflQAVa4L88qRvBXPM0q4AOwBF+U/1pZdYoQWW70LAUWlzZ2TWNnAOyMEMC54ziYXzy6N0jMlW8US9hFfN8bL5LBoMBuZTZRYnf/tvP6UnnHBNYczQ8PbHKr81wSu5B84GKdfPd/wUvLlenJ8YsYYEbrem7ZMXXoszBXLhCfzAcAud6mXIwCUAkLVkm8XzkDM5XVtiZmxwkyCcE0BMMUVZg+ZWiIwASJPyCc4MNUq+xAnMD1YDtkRSQXLsIA9keLFlDR6QwwJsYGQ3De4EZBWDz+VF30cxluQDbuBAKZFZwZNTm7FGHY5ODr8LTYXDALKB/WMCIEAXsCuNwWC/Q0FmKNTdzXQyq2dKfUXjfbqHBkqjGEjO6GXX3ALnN7EP4rlcaheOgPaqB9IadZtRd0EA10FhsAP3LetmKX6Nn6SjHSMcV/KiWSq2cT48yvY4PcHYwDHr2idtw5Dly9JDysiSJKCshY0Buy6/aeN8KaCbf2DWOeAkc7IwyvjASMuHXbbUOlb72pydPzx+9+/HNOR6rFJih5pSFl3xXUl8hzKsf30X1MFJDUE+8g4iEow0g9Ix2Axs7+sn5yydvXn3/5tEP3dHt2EXj3elJwheEJUkEERn0s4ImC9/EDrt1AjSSxQwBknvzEHx0WOYgSOoNvIDwa0Tu6btiw/0QgpZdzikcMOMQcd4OHJM31/gG1GbDedOnzWJe942cvoSBnJNr6LXzyJ/IvW/GYJhUjwW/PfKxcWTbxrrNhmsgRgm0mk1egIdsThq1kg4g3rrgaWk3DW7sNXhDE9ZqIx08GCUEIzrww+iZrgWgGERbxlAGMCqBME7FwXG8KVi8C5U7VM4YjYHLekHDoVhgz5QMnUDNcC3zUDKpmzVzSEJ9ORQzHP8M+6GFmjHlZq2MzS6jR8BSJCuaX/LTC2dFAj5/im2sWJYA4N0dLKDXNs3UcKPBtGa31Rxv1ppenMkYDErC/5SCbgPVuFkj36lvOVeBF8TGUzIGPePkgWYGcYxKUDsYAOheYjtuysxTmC8qlYIcX26L/onCl0QsFnThk4dk6KOdAdmAzOTcsqHMRhgBO7RnWjBgDNRwOnN65/68UYAl0avsRZoxEC0s7aP9hsOeafL4tMFojlKo57W6od8QAcq4Nfsb5pyRsYMAawgHqT0CeYF7XXidtHAC3jq7vGDx5fSmHdbsd5A2gDB4FT2f3tjYZh8Qzwm0ycJbixqzohI83/SmG/DsNSw1XHgvMpZgSlanSGEYeo2qBR6vPAkLzgDDyiuqArszgH473gtcJtofCEEgXSunM6/OCAHXNNwF2kXh3wr8DvxRLsKbdxZvgadq7AzQnOAJinRySg+WM2jnwzZKEKJyMZjK5BpcrUjuOhnj5/JEQoF5iMWsJ7fACAZb86Ty9YsdddjxLWiHaNnIkGRrTj3056qjd3NrzFwo0gY45YsqyuS0tccCQzy9L7ulAyoz/a91Gg9tc7M2Inxnw2D686NSc42ZkBtEYa2gSbAnCRMQmRbZFmLUKxDbjV7tTs32nXkoN1L8Y8Opv7dVjQP71Fk4WmiRpSn+NYEN+gy6AMTtSytrKO1a88xrpiv3R4+qKiBoU9OnLC25r+d58+O2bj+h4ZM+AKDDEwk2I7KknD4Y+dp13IaGI/GUX/F0OgxII3rHYjGfOqw+NHbqk9smLb1CPyBdR1CNYM2B+JTVh2y5pM2+u1HnlFoMwCbPdyAlUnbkBuIw2AJHa3SLm1lfBI/6tt2HdD0VBH19PxzfdymFKHbwQelmLRHJ+iYG4L3ylMW8iyeoHVtoAREARRBmSehhlW2ghh3TlqZnKXUl3cE7E/l1hb6Oqk9Keq2240gXjBUGEJPqZRVLbbGIxUUGCQEWOxA4SoLnUAdvuLbRR8uKHaEcZE3oYji71JmMLmoNYIt4rJ22XvU6qZc6zUZER6BXieA2h9g5iijElK0vEkbARZcat7G4Fq7Rp7tnsbfeNiVYcDMSU+yV7Z0cJFH/p42UVUIxTqHsoqSwG7AH2A4iPh98XYeJlsuedPJWfN4WNm/HqmXT5dJK/7AyeuZURiGdknGW8ATrXLaeimYbiyLepJB9VjtzymzdA2IGlT3ZLaZZPLuVWkKYqcxk3lA08kOaf4yiFmAfSYzdXWbD5rR1iGhAcJgxMxumbjfXGkNPEUknmhC0Qx/FjAK+5gL+ak7PyGjcsaieGlNNBPpuR6S3YjI/zkn+KU76iPRxUhOpEU3XLhDHNECCqSoL/JKMh76qiGWLBYiYXHKel3iHkdT3OGzLIGDBouuwA3GtiojrvHp92xmGpIf9qqozV5yEw45r6inwbzm/BBejYnETvcl094ko4kjgYHP3hnRP5GAW6wSoJg8IZVZJBmf9i77QzfbOQUP5jrYIzmyJUEOUbTR0MfIddQoqbcG3CoxIQlM4ckTrtQw2WWqWiqrQWhqzHg6Bnc75OlYxU9tx6R5hyez0j6Gug796I+FCpCnIGJJfF2eNif/+62+1/TiXOIS+xoggk38m8YozwN2Y5fryTFe8l1hasN7RP2I9+UGJqGcpx5Awx1ZFZLr0nWQesnxMSpY++Y7cn7TFYTL5oc3sVU0E5Lvs2JmmOutkpw6FWBWBsgrTZFGCvGKfcMgZGtIdH9oqen/GvEKW5+mOqrGODvouylQUdayydmaY8Ep1aed0f14DVvT1Su4RbhrxrmPfus3V+PRg9LkT1JK3PUSqpAH+GUAwIEsIsD9Bo5WqOXpumJsBGRR1p0rZVbYuIARoBM2x1o0dn+FcKsx1ieQt5Hm8pM2anRm9FxLqRPfSPLv16e1IG8JulYvQYWAqju1Y7iChcVY9msg8BShQV6dYflD5/Ev20rxBULebihzYViIY+DRaMHAfPIHYPmZYyk3FJa+9ZZoOfuaFvRHCqik2yaw+5CtRVlmxq0uwsIfogi/QJdVO0Bb55/rQmdzazbBVbbaZ6thFU7RAiENzv8EGd7S/xx2SmKfp0W16LTBoc9XHqMHTg3ZNjrog+5ZdcffdRTtnaKZWmarKubcencLMR/lTkUEGf+OO2RNCVb3GFAnq+kugyzg1rX1TSvG6wYNgS5mVlYgnoLAlxLADC84AS/a9SM5EcfyutwU+H+XT1iQl9V+Q5i9YPg6/wdcCshRYOEcbxqckZ0m2lfqGE375E6uSeN2nAIXlZgSyDGNmdDEKyGIMfmqt4nH9c96p0iASCegYIRDhjT3H4IB2ZO6OXoxxqB0wE5iyTDouGSwUk6/Oynqz89qIkVPHeBVxyzOugdjHE4rDAggwd1OTOl1PyADJX8+AqN9xf07dbjGajO+Xe/Ld4CH8GOsf5CaefBneW+ytRNGlHRwhmE4lFjMwxLUqxzhXlOCjJHiQZCMTvNRm6D9E6bzr6NpTWeE7hSIR6oJU6xS+qAdAg/+giPVRvq0p4hG3NPV3RdY8Y3Dc2ft+s2kiTBxA30P+/77OmCHwfm9SVJWWPrDBqL3eKdiOzsxd5Wwx1xp0rcNa23uc8t44HEg2trr5g7rvVKzS91t8dlOtsmTqbWEPrhSf1DeQeFJSTG7bz6QsNKQVYNESdEXhj7opoGPwFt82d0MwOXIjiw+zAa0fVeA9YED0hSBsF2207ntIRib6sZGZ5Yl+wAc+kb65Uv4l6C407fx2+GXXK6w/0s4IQ2LqwQo7iAOAsjfAy1OW5is2HYb3HRqVqFLYstd5UwYZTvOOrn7q1RGhe/Hiew7RndoQ9V6Yp0d4S85kzL3WwhjvpWyXbSrqtOPTHtAAbV8sByTJxfTboTMwTrOS0w6yo49o7aPKNMo3pPaueTyTohLNw0DjUupnR9P6MQ9qqpra7YKIC4EPCfHSAdJfNcjrptB6popn2vTRvEond61HajOHyHYjqxI9N97g1MXCtgvrsG6I0pZSJqTnPmx6o4bqWh2gmna1datxvJNwuNgfTmfX9UAsvajp7mQ9ddRgZAOqaleQdRzewE3IDdXsf3fw5MyEL/suNKr7HwLItWvUpV6nmZdpzou0z2AjXlkVmVDROoWg9Qmr2NMCr6AQWPQFkgkT1Q91vlSgeMTV1SW7ElM7atibWjaPjK5jeGsNfiscMsGQCuFx01Ot8Nkxuc01sxANqWchTfyHs3vivo8S3wKquE9dWsEo0O5x+U7cSE/VupXj7VBX1/r4DguFWwtah9tqifbQRLuHFB/6LUPMDvRrwtLi/8AhgvFBFTcJCrsW5XTUoajviIGsoaOEMlORrb5Mh0OQ+P68mwgrdeMKPad5CnsNSOt862WOhRP2iN5U8eResof4YKqCa61B2IOPzT7Bp8HOYEWzZfj1w8jmPaTFnDXDCuYN4P8EVLRFt6vUAt8Dgg6rFaSZ+OgQgOzto+/vj6PZ4TRYnSoeamYVc9H/BOmeIqmJtkuHLJhnbZldnDJwotDOmXKAea2+zyNfa7jCv4OQzX3XmhcZ5Dec/PHXkloAeraV3m0QIoQhMqFjv0UCrL7kRYXZr4fMQyzgyrETIr52dlCnLIYXSGK3gars/KxfHidNfqEzwiTM8kq9FMskvuSjHnh/yJ93KLDIICcw8BJCZHyIYDq3IqlW0DweDv0DdLLbgMwO3yLI5WFeqLF2DeDQTgsdk2tf2vtunmjfu1m411WTTySNdsLRpNG3j90AcaMIHzREESKHF0X48CWKvIm9exCqtvNvW892FRQxAAA=",
    "sha256": "99acf9da8e13b252dc123f4b2f871458f33b13cdb9c89d889a8f40f5b7a113ff",
    "bytes": 12564
  },
  "postprocess_submission.py": {
    "payload": "H4sIAMO+ImoC/+1Y3W7iRhS+R+Idpo4q2V1wIZCVFi2VVptEu1KyWSWkF6UIGXxMrBgb2WNYirjtA/QR+yQ9Z35sg23YXlaquRnPfOd3zh/2l6so5myerJsNX66TbZKtI1zSz423cRqyIbt1ggTw2GNGu427BvNDorCdeLEeNBsMnww8ilOQWxphx7CM1mAqYot4pwlMl06YOsGUv8SQvESBmxyLkoAqaXXkJ4Urdpa07vOXr8+j6e3nuxuk0+hxd9Js3D/fjT5/FfteEDnczE4vJxZD1QIIsz2L/cIuGaDWrGO/azZGnx5vnj493F1Pbx8ep7/dPD6U2fSq2fQ0m06z8fHu4enmy83T0zRjWObTr+bT13z678jQC+ZGkIScbaL4lW2Bt5izWgU+uGwGXhQDS+ZO4IcLgt5/+PL84S4X+oRSx2jYVYt17bctdmn3WqxnX7ZY3+5MTlC8xYdorq7KRGdI3goSIuzZfU3TbDw8jwo35hm7/AbtBO3hpmFjSBvWuDPZT3f6FnFZvhPcrPDwXtCL6KsMMBV8/06Re8W02VjFfohnT9zhiXL9gKJxHgXpMqTw3e0Jt/H5C4tWeKs54xYLYYOXBEPDsJiTME8pE4PjQoy0KMW+9uf8UWyYniXPUQqLow3lkIQqOn32CtsWWzuBAEQb2+ewTEyrgKIHPYJANhwyI4aFH4VT3zWOMPTMo5D7oc7BfFcYaCfAXfCcNOCmEDueWDaGIoSuKSMb9bBEejpBMMWXFMgr40nmu68QtyU35qwhdhaQCBeSJbgvLCEiNEYLPTTIWS8o3dOlKZEW+1nkj3pToEy6Dd84qXdwLHXxDMZ2KGQ/YDvkOrD73l7Wlog0CwIlJ2emZRV2tGGe8aCIlFnIVLEpMFZOuPYTHvuzlOM9ZNb74gKdcAFmt6OtzUIXdenYV+wn5qv6iV6noMkBbwggD1G3ZDoDvgGggk4qj9ciWNYkpHA3GBY5h/d4/H4oWU+OXLXLYPv2TiDIbQeCSr5BbbudzsDuefsfddHmEceMdH2sWLELroSScZhFFEnudA5BkMgdx/NgznFPhqzaRWQQJRBCktQwCiI8pONl5Pqef8j0O9PTDz0/gJbEFQpGixkbo4SOUk7wqpSWL6bkp7y6iTGoNUC+mIqF8NO5vEfpUzqTqaV3VWJrr+XNuFguyBOqVBylP4+3FQXhTCk9ftZZkyNBVjXooOMHmAxmqaHUUF6wVeDMgWjRfw5HVsAwv4VpDpsH4IR4IAzd4ts5+bp6YZx2OjUyRXK2kIjcBmG6xKzmYOZMrBpfKPdhUiHxCYy8O/KFfxo0w0B4LUNoWKhhL9kWLgRzUvfVagpUmIjes3LPPWFCdVq/GRKveqqDjEdwtx4qDemcdHQY8eMcOOP1UoE5rUR1kuUjazlal84rYA0FjHRRtNBRzAkxkDgsME75C4awfjFd7L046qWzALD5paGMbqpl7A+IIz3t1YQpBOgCZ5aY5Ko25neKcY1ry8LLrJiVTrimvr6ifypFnORVXYy/575zCWWkqoE6g8sg+DaHFWe/kuI3cRzFg7NMZIrgvb0CrDCewrZId3/OpM5m4ON9vohynFDhb/uF8JaFXNXzaGMq5lZpdHQ8BFROjtRQ6UUy7QxYpyWX3Xx5mS97+bKfL6/ypZGGwoeAw57YPJpPDxrbf3hA9QKe1TmaQUu8BWDIxDAT8KqKfZG3BUxrzNMZqFStaIohX5BHNLsygi5yLGCTikivqdgXsoTJC6vhWbjRKs7/T+rnJ/Xfw++Z1WVLUwmprEhsycy0rNPD/GtLjtqC5tAt2UT9KiZoVLQgSU3Ml9nErNraqRFMMxzpAz9cMPnRxGW7ys4sJJB+4hSH3LgGeKDdwL5CKvrfIc6Mozv6oFuiaqZ0WUf9dd/KWq6sqIgpDgH7Y54fdScq/E3JVETaukYlDZQmODge6t4jSCr70V4qdKzANaJ/YPeYR7784kJhoyJ/tmX5Rwohhto0YtQ5ZSk6tuLjhQCrNNZoYYsYCUJW+W2D/f3nX2xXqNjCW4VScqjzqFDNUFMZPcXRe1causeDdndSI0cFo/xKqARGif44V0AfONC4ll8V28yFAOja8W/OKuWM/uoQ238Axn+QbMoUAAA=",
    "sha256": "27b89bc8ecc16e8a2369ed695b3130efb4f9f4d7faf11303491f5f1fb99b1cbb",
    "bytes": 5322
  }
}

for name, meta in SCRIPT_BUNDLES.items():
    raw = gzip.decompress(base64.b64decode(meta['payload']))
    digest = hashlib.sha256(raw).hexdigest()
    if digest != meta['sha256']:
        raise RuntimeError(f'sha256 mismatch for {name}')
    out_path = WORK_DIR / name
    out_path.write_bytes(raw)
    print(f'wrote {out_path} ({len(raw):,} bytes)')


wrote /kaggle/working/deep_drought.py (102,862 bytes)
wrote /kaggle/working/cluster_regions.py (12,564 bytes)
wrote /kaggle/working/postprocess_submission.py (5,322 bytes)


## Stage A Region-Level Dataset

`TRAIN_FRACTION` and `MAX_REGIONS` select complete regions, not random rows. That keeps each selected region's daily timeline intact.

In [4]:
def _read_csv(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

raw_train_path = INPUT_DIR / 'train.csv'
raw_test_path = INPUT_DIR / 'test.csv'
raw_sample_path = INPUT_DIR / 'sample_submission.csv'

train_full = _read_csv(raw_train_path)
test_full = _read_csv(raw_test_path)
sample_full = _read_csv(raw_sample_path)

all_regions = np.array(sorted(train_full['region_id'].astype(str).unique()))
rng = np.random.default_rng(SEED)
if TRAIN_FRACTION <= 0 or TRAIN_FRACTION > 1:
    raise ValueError('TRAIN_FRACTION must be in (0, 1]')
n_regions = int(math.ceil(len(all_regions) * TRAIN_FRACTION))
if MAX_REGIONS is not None:
    n_regions = min(n_regions, int(MAX_REGIONS))
min_regions = 2 if len(all_regions) >= 2 else 1
n_regions = max(min_regions, min(n_regions, len(all_regions)))
selected_regions = np.sort(rng.choice(all_regions, size=n_regions, replace=False))
selected_set = set(selected_regions.tolist())

train = train_full[train_full['region_id'].astype(str).isin(selected_set)].copy()
if RECENT_YEARS_PER_REGION is not None:
    years = train['date'].astype(str).str.slice(0, 4).astype(int)
    train['_year'] = years
    max_year = train.groupby('region_id')['_year'].transform('max')
    train = train[train['_year'] >= max_year - int(RECENT_YEARS_PER_REGION) + 1].copy()
    train.drop(columns=['_year'], inplace=True)

if SUBSET_TEST_TO_SELECTED_REGIONS:
    test = test_full[test_full['region_id'].astype(str).isin(selected_set)].copy()
    sample = sample_full[sample_full['region_id'].astype(str).isin(selected_set)].copy()
else:
    test = test_full.copy()
    sample = sample_full.copy()

train.to_csv(DATA_DIR / 'train.csv', index=False)
test.to_csv(DATA_DIR / 'test.csv', index=False)
sample.to_csv(DATA_DIR / 'sample_submission.csv', index=False)
pd.Series(selected_regions, name='region_id').to_csv(WORK_DIR / 'selected_regions.csv', index=False)

print(f'Raw train regions: {len(all_regions):,}')
print(f'Selected regions: {len(selected_regions):,}')
print(f'Staged train rows: {len(train):,}')
print(f'Staged test rows: {len(test):,}')
print(f'Staged sample rows: {len(sample):,}')
if TRAIN_FRACTION < 1.0 or SUBSET_TEST_TO_SELECTED_REGIONS:
    print('NOTE: subset mode is for validation/ablation, not a full Kaggle submission.')


Raw train regions: 2,248
Selected regions: 2,248
Staged train rows: 12,075,916
Staged test rows: 204,568
Staged sample rows: 2,248
NOTE: subset mode is for validation/ablation, not a full Kaggle submission.


In [5]:
# Patch only the temporary /kaggle/working copy of cluster_regions.py for small subsets.
def patch_cluster_script_for_subset(n_selected_regions: int):
    path = WORK_DIR / 'cluster_regions.py'
    src = path.read_text(encoding='utf-8')
    if not AUTO_ADJUST_CLUSTERING_FOR_SUBSETS:
        return 30, 30
    if CLUSTER_TARGET_K is None:
        target_k = 30 if n_selected_regions >= 300 else max(2, min(30, n_selected_regions // 10))
    else:
        target_k = int(CLUSTER_TARGET_K)
    target_k = max(1, min(target_k, n_selected_regions))
    if CLUSTER_MIN_SIZE is None:
        if n_selected_regions < 60:
            min_size = 1
        else:
            min_size = 30 if n_selected_regions >= 900 else max(2, min(30, n_selected_regions // max(target_k * 3, 1)))
    else:
        min_size = int(CLUSTER_MIN_SIZE)
    min_size = max(1, min(min_size, n_selected_regions))
    src = re.sub(r'LOOKBACK_YEARS\s*=\s*\d+', f'LOOKBACK_YEARS   = {int(RECENT_YEARS_PER_REGION or 10)}', src)
    src = re.sub(r'TARGET_K\s*=\s*\d+', f'TARGET_K         = {target_k}', src)
    src = re.sub(r'MIN_CLUSTER_SIZE\s*=\s*\d+', f'MIN_CLUSTER_SIZE = {min_size}', src)
    path.write_text(src, encoding='utf-8')
    return target_k, min_size

cluster_k, cluster_min_size = patch_cluster_script_for_subset(len(selected_regions))
print(f'Cluster settings for staged data: TARGET_K={cluster_k}, MIN_CLUSTER_SIZE={cluster_min_size}')


Cluster settings for staged data: TARGET_K=30, MIN_CLUSTER_SIZE=30


In [6]:
# Run clustering once. deep_drought.py will consume /kaggle/working/region_clusters.csv.
cluster_log = LOG_DIR / 'cluster_regions.log'
cmd = [sys.executable, '-u', 'cluster_regions.py']
print('Running:', ' '.join(cmd))
started = time.time()
with cluster_log.open('w', encoding='utf-8') as fh:
    proc = subprocess.run(cmd, cwd=WORK_DIR, stdout=fh, stderr=subprocess.STDOUT, text=True)
elapsed = time.time() - started
print(f'cluster_regions.py exit={proc.returncode}, elapsed={elapsed:.1f}s, log={cluster_log}')
tail = cluster_log.read_text(encoding='utf-8', errors='replace').splitlines()[-25:]
print('\n'.join(tail))
if proc.returncode != 0:
    raise RuntimeError('cluster_regions.py failed; inspect logs/cluster_regions.log')


Running: /usr/bin/python3 -u cluster_regions.py
cluster_regions.py exit=0, elapsed=46.6s, log=/kaggle/working/logs/cluster_regions.log
5          48               -0.50               -0.87                -0.59          0.34                  0.80             1231.56              24.51                  0.58                 -0.78                0.19             14.78              0.62                     1.57          35.40
6          56               -0.64               -0.73                -0.62          0.48                  0.89             1220.49              20.84                  0.50                 -0.64                0.23             19.35              0.40                     2.32          31.05
7         175               -0.50               -0.87                -0.60          0.47                  0.91             1303.34              21.53                 -0.00                  1.00                0.20             17.55              0.45                     2.28          3

## Launch Ablation Training Runs

The current training script is single-GPU. On Kaggle T4 x2, this cell runs up to `MAX_PARALLEL_RUNS` independent training processes at once and assigns one visible GPU per process.

In [7]:
def _cfg_args(cfg: dict) -> list[str]:
    args = []
    for key, value in cfg.items():
        args.extend(['--cfg', f'{key}={value}'])
    return args

def build_run_command(run: dict) -> list[str]:
    name = run['name']
    shared_cfg = {
        'run_name': name,
        'data_dir': str(DATA_DIR),
        'checkpoint_dir': str(CHECKPOINT_DIR),
        'submission_path': str(SUBMISSION_DIR / f'{name}.csv'),
        'n_epochs': N_EPOCHS,
        'samples_per_region_per_epoch': SAMPLES_PER_REGION_PER_EPOCH,
        'batch_size': BATCH_SIZE,
        'num_workers': NUM_WORKERS,
        'use_tta': str(USE_TTA).lower(),
        'seed': SEED,
    }
    merged = {**shared_cfg, **run.get('cfg', {})}
    return [sys.executable, '-u', 'deep_drought.py', 'train'] + _cfg_args(merged)

def parse_log_metrics(log_path: Path, target_epoch=None) -> dict:
    text = log_path.read_text(encoding='utf-8', errors='replace') if log_path.exists() else ''
    epochs = []
    current = None
    for line in text.splitlines():
        m_epoch = re.search(r'epoch\s+(\d+)', line)
        if m_epoch:
            current = {'epoch': int(m_epoch.group(1))}
            epochs.append(current)
        if current is None:
            continue
        m_val = re.search(r'val_mae=([0-9.]+).*?per-week:\s*(.*)', line)
        if m_val:
            current['val_mae'] = float(m_val.group(1))
            for wk, val in re.findall(r'w(\d+)=([0-9.]+)', m_val.group(2)):
                current[f'val_mae_week{wk}'] = float(val)
        m_weighted = re.search(r'val_mae_w\s*=\s*([0-9.]+)', line)
        if m_weighted:
            current['val_mae_weighted'] = float(m_weighted.group(1))
        if 'per-week val_mae_weighted:' in line:
            for wk, val in re.findall(r'w(\d+)=([0-9.]+)', line):
                current[f'val_mae_weighted_week{wk}'] = float(val)
    if not epochs:
        return {}
    if target_epoch is not None:
        for e in epochs:
            if e.get('epoch') == target_epoch:
                return e
    return epochs[-1]

def summarize_run(run: dict, returncode: int, elapsed_sec: float) -> dict:
    import torch
    name = run['name']
    log_path = LOG_DIR / f'{name}.log'
    ckpt_path = CHECKPOINT_DIR / f'{name}_best.pt'
    if not ckpt_path.exists():
        eps = sorted(CHECKPOINT_DIR.glob(f'{name}_ep*.pt'))
        ckpt_path = eps[-1] if eps else None
    summary = {
        'run_name': name,
        'purpose': run.get('purpose', ''),
        'enabled': run.get('enabled', True),
        'status': 'ok' if returncode == 0 else 'failed',
        'returncode': returncode,
        'elapsed_sec': round(float(elapsed_sec), 2),
        'log_path': str(log_path),
        'checkpoint_path': str(ckpt_path) if ckpt_path else '',
        'cfg_diff': json.dumps(run.get('cfg', {}), sort_keys=True),
        'train_fraction': TRAIN_FRACTION,
        'selected_regions': len(selected_regions),
        'n_epochs': N_EPOCHS,
        'samples_per_region_per_epoch': SAMPLES_PER_REGION_PER_EPOCH,
        'batch_size': BATCH_SIZE,
    }
    target_epoch = None
    if ckpt_path and Path(ckpt_path).exists():
        try:
            ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        except TypeError:
            ckpt = torch.load(ckpt_path, map_location='cpu')
        target_epoch = ckpt.get('epoch')
        summary.update({
            'best_epoch': target_epoch,
            'val_mae': ckpt.get('val_mae'),
            'val_mae_weighted': ckpt.get('val_mae_weighted'),
            'best_metric': ckpt.get('best_metric'),
            'best_metric_name': ckpt.get('best_metric_name'),
        })
    summary.update(parse_log_metrics(log_path, target_epoch=target_epoch))
    return summary

enabled_runs = [r for r in ABLATIONS if r.get('enabled', True)]
print('Enabled ablations:', [r['name'] for r in enabled_runs])


Enabled ablations: ['baseline', 'no_daily', 'no_long_context', 'no_cluster', 'no_region_embedding', 'replicate_weeks', 'no_training_tricks', 'extra_features']


In [8]:
results = []
pending = list(enabled_runs)
active = []
max_parallel = max(1, min(int(MAX_PARALLEL_RUNS), len(GPU_IDS)))
print(f'Launching up to {max_parallel} parallel run(s) on GPU ids {GPU_IDS}')

while pending or active:
    while pending and len(active) < max_parallel:
        run = pending.pop(0)
        gpu_id = GPU_IDS[len(active) % len(GPU_IDS)]
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
        log_path = LOG_DIR / f"{run['name']}.log"
        fh = log_path.open('w', encoding='utf-8')
        cmd = build_run_command(run)
        print(f"[{datetime.now().strftime('%H:%M:%S')}] start {run['name']} on CUDA_VISIBLE_DEVICES={gpu_id}")
        fh.write('COMMAND: ' + ' '.join(cmd) + '\n')
        fh.write(f'CUDA_VISIBLE_DEVICES={gpu_id}\n')
        fh.flush()
        proc = subprocess.Popen(cmd, cwd=WORK_DIR, stdout=fh, stderr=subprocess.STDOUT, env=env, text=True)
        active.append({'run': run, 'proc': proc, 'fh': fh, 'started': time.time(), 'gpu_id': gpu_id})

    still_active = []
    for item in active:
        proc = item['proc']
        rc = proc.poll()
        if rc is None:
            still_active.append(item)
            continue
        item['fh'].close()
        elapsed = time.time() - item['started']
        run = item['run']
        print(f"[{datetime.now().strftime('%H:%M:%S')}] done {run['name']} rc={rc} elapsed={elapsed/60:.1f} min")
        results.append(summarize_run(run, rc, elapsed))
        pd.DataFrame(results).to_csv(WORK_DIR / 'ablation_results.csv', index=False)
    active = still_active
    if active:
        names = ', '.join(item['run']['name'] for item in active)
        print(f'active: {names}; waiting...')
        time.sleep(30)

results_df = pd.DataFrame(results)
results_df.to_csv(WORK_DIR / 'ablation_results.csv', index=False)
results_df


Launching up to 2 parallel run(s) on GPU ids ['0', '1']
[14:12:41] start baseline on CUDA_VISIBLE_DEVICES=0
[14:12:41] start no_daily on CUDA_VISIBLE_DEVICES=1
active: baseline, no_daily; waiting...
active: baseline, no_daily; waiting...
active: baseline, no_daily; waiting...
active: baseline, no_daily; waiting...
active: baseline, no_daily; waiting...
active: baseline, no_daily; waiting...
active: baseline, no_daily; waiting...
active: baseline, no_daily; waiting...
active: baseline, no_daily; waiting...
active: baseline, no_daily; waiting...
active: baseline, no_daily; waiting...
[14:18:11] done no_daily rc=0 elapsed=5.5 min
active: baseline; waiting...
[14:18:41] start no_long_context on CUDA_VISIBLE_DEVICES=1
active: baseline, no_long_context; waiting...
active: baseline, no_long_context; waiting...
active: baseline, no_long_context; waiting...
[14:20:11] done baseline rc=0 elapsed=7.5 min
active: no_long_context; waiting...
[14:20:41] start no_cluster on CUDA_VISIBLE_DEVICES=1
act

,run_name,purpose,enabled,status,returncode,elapsed_sec,log_path,checkpoint_path,cfg_diff,train_fraction,...,val_mae_week1,val_mae_week2,val_mae_week3,val_mae_week4,val_mae_week5,val_mae_weighted_week1,val_mae_weighted_week2,val_mae_weighted_week3,val_mae_weighted_week4,val_mae_weighted_week5
0,no_daily,Remove 91-day Transformer branch,True,ok,0,330.01,/kaggle/working/logs/no_daily.log,/kaggle/working/checkpoints/no_daily_best.pt,"{""use_daily_branch"": ""false""}",1,...,0.514,0.521,0.529,0.540,0.547,0.689,0.694,0.705,0.709,0.718
1,baseline,Full current model,True,ok,0,450.06,/kaggle/working/logs/baseline.log,/kaggle/working/checkpoints/baseline_best.pt,{},1,...,0.497,0.509,0.520,0.538,0.552,0.656,0.665,0.677,0.691,0.703
2,no_long_context,Remove engineered long-context features,True,ok,0,570.04,/kaggle/working/logs/no_long_context.log,/kaggle/working/checkpoints/no_long_context_be...,"{""use_long_context"": ""false""}",1,...,0.513,0.520,0.529,0.540,0.550,0.693,0.699,0.708,0.716,0.723
3,no_cluster,Remove cluster features and embedding,True,ok,0,570.07,/kaggle/working/logs/no_cluster.log,/kaggle/working/checkpoints/no_cluster_best.pt,"{""include_cluster_aggs"": ""false"", ""mixup_withi...",1,...,0.501,0.510,0.521,0.532,0.549,0.661,0.669,0.685,0.695,0.701
4,no_region_embedding,Remove region identity embedding,True,ok,0,570.04,/kaggle/working/logs/no_region_embedding.log,/kaggle/working/checkpoints/no_region_embeddin...,"{""use_region_embedding"": ""false""}",1,...,0.489,0.502,0.515,0.529,0.545,0.651,0.663,0.677,0.692,0.699
5,replicate_weeks,Predict the same value for all 5 future weeks,True,ok,0,570.06,/kaggle/working/logs/replicate_weeks.log,/kaggle/working/checkpoints/replicate_weeks_be...,"{""replicate_prediction"": ""true""}",1,...,0.511,0.522,0.535,0.547,0.562,0.639,0.650,0.662,0.673,0.688
6,no_training_tricks,"Remove mixup, label noise, and curriculum",True,ok,0,570.04,/kaggle/working/logs/no_training_tricks.log,/kaggle/working/checkpoints/no_training_tricks...,"{""use_curriculum"": ""false"", ""use_label_noise"":...",1,...,0.536,0.546,0.560,0.576,0.594,0.644,0.653,0.660,0.673,0.689
7,extra_features,Turn on extra engineered features,True,ok,0,600.04,/kaggle/working/logs/extra_features.log,/kaggle/working/checkpoints/extra_features_bes...,"{""extra_features"": ""true""}",1,...,0.485,0.494,0.500,0.508,0.527,0.643,0.648,0.665,0.670,0.675


In [9]:
# Quick ranking view
results_path = WORK_DIR / 'ablation_results.csv'
df = pd.read_csv(results_path)
metric = 'val_mae_weighted' if 'val_mae_weighted' in df.columns and df['val_mae_weighted'].notna().any() else 'val_mae'
ranked = df.sort_values(metric, na_position='last').reset_index(drop=True)
ranked.to_csv(WORK_DIR / 'ablation_results_ranked.csv', index=False)
print('Ranking metric:', metric)
ranked[['run_name', 'status', metric, 'val_mae', 'best_epoch', 'elapsed_sec', 'cfg_diff']]


Ranking metric: val_mae_weighted


,run_name,status,val_mae_weighted,val_mae,best_epoch,elapsed_sec,cfg_diff
0,extra_features,ok,0.6600,0.5028,2,600.04,"{""extra_features"": ""true""}"
1,replicate_weeks,ok,0.6623,0.5354,2,570.06,"{""replicate_prediction"": ""true""}"
2,no_training_tricks,ok,0.6633,0.5622,2,570.04,"{""use_curriculum"": ""false"", ""use_label_noise"":..."
3,no_region_embedding,ok,0.6761,0.5160,3,570.04,"{""use_region_embedding"": ""false""}"
4,baseline,ok,0.6780,0.5233,4,450.06,{}
5,no_cluster,ok,0.6818,0.5226,5,570.07,"{""include_cluster_aggs"": ""false"", ""mixup_withi..."
6,no_daily,ok,0.7027,0.5301,8,330.01,"{""use_daily_branch"": ""false""}"
7,no_long_context,ok,0.7074,0.5306,8,570.04,"{""use_long_context"": ""false""}"
